In [0]:
#standardization and replacement
#standardization1 - Column Enrichment (Addition of columns)
#create new column with default value
from pyspark.sql.functions import col,lit,upper

standdf1=spark.read.csv(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/staging/custs",inferSchema=True).toDF("custid","fname","lname","age","profession")
standdf2=standdf1.withColumn("source",lit("raw"))
#display(standdf2)

#Standardization2 - Column Uniformity
#profession column Uniformity with upper
standdf3=standdf2.withColumn("profession",upper("profession"))
#display(standdf3.limit(20))

#Standardization3 - Format Standardization
#check id and age column if it contains non integer values
standdf3.where("custid rlike '[a-zA-Z]'").show()
standdf3.where("age rlike '[^0-9]'").show()

standdf3.printSchema()

#replace ten with 10 using replace function 
#regreplace of age by removing - between 4-7
from pyspark.sql.functions import replace,regexp_replace
replacedict={'one':'1','two':'2','three':'3','four':'4','five':'5','six':'6','seven':'7','eight':'8','nine':'9','ten':'10'}
standdf4=standdf3.na.replace(replacedict,["custid"])
#standdf4.where("custid rlike '[a-zA-Z]'").show()
standdf4.where("custid='10'").show()

standdf5=standdf4.withColumn('age',regexp_replace(col('age'),"-",""))
#standdf5.where("age rlike '[^0-9]'").show()
standdf5.where("age=47").show(10,False)

In [0]:
#Standardization4 - Data Type Standardization
standdf5.printSchema()

standdf6=standdf5.withColumn("custid",standdf5['custid'].cast('long'))
standdf6=standdf6.withColumn("age",standdf5['age'].cast('short'))
standdf6.printSchema()

In [0]:
#Standardization5 - Naming Standardization
#rename the existing columns

standdf7=standdf6.withColumnsRenamed({'custid':'CustomerID','fname':'FirstName','lname':'LastName','age':'Age','profession':'Profession','source':'Source'})
display(standdf7.limit(10))


In [0]:
#Standardization6 - Reorder Standadization
#original column order in dataframe
#display(standdf7).limit(10)
#reorder
#display(standdf7.select("Age","Profession","FirstName","LastName","CustomerID","Source").limit(10))

#drop column using na.drop
standdf8=standdf7.na.drop(subset=['profession']) #this will remove only the null value present in the profession column
display(standdf8.count())

standdf9=standdf8.drop("Source")
display(standdf9)
